# THIS IS THE FINAL VERSION FOR nnUNET_basic_SCV
all of the necessary splitting, Simple Straificaion, Cleaning, and ROI Masking has been done before this NB

## Importing all the Dependencies

In [1]:
# IMPORT NN UNET

import sys

sys.path.insert(0, "/kaggle/input/notebooks/kalabhalu/nnunet-importer")

# Checking with INTERNT OFF and GPU ON
import nnunetv2

import importlib.metadata as md

print(md.version("nnunetv2"))

2.8.0


## Importing the properly unzipped notebook to process

In [2]:
from pathlib import Path
import random

sys.path.insert(0, "/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper")
base = Path("/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper")

files = [f for f in base.rglob("*") if f.is_file()]

for f in random.sample(files, min(10, len(files))):
    print(f.relative_to(base))

nnUNet_raw/Dataset001_CAC/imagesTr/d25568e71932_0000.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTr/345f785de333_0000.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTs/895364818b79_0000.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/62d8b139ed18.nii.gz
__results__.html
nnUNet_raw/Dataset001_CAC/imagesTr/30ed745e8552_0000.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTr/f9f3f4b95343_0000.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTr/d626ce1d0f14_0000.nii.gz
nnUNet_raw/Dataset001_CAC/labelsTr/52712d30f9e2.nii.gz
nnUNet_raw/Dataset001_CAC/imagesTs/c4d390538416_0000.nii.gz


## Environment Variables for nnUnet

In [3]:
import os
import nnunetv2

import importlib.metadata as md

print(md.version("nnunetv2"))

os.environ["nnUNet_raw"] = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/kaggle/working/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/kaggle/working/nnUNet_results"  

!mkdir -p /kaggle/working/nnUNet_raw
!mkdir -p /kaggle/working/nnUNet_preprocessed
!mkdir -p /kaggle/working/nnUNet_results

2.8.0


## Pre Prcoessing for nnUnet

In [4]:
import sys
from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry

sys.argv = [
    "plan_and_preprocess",
    "-d", "1",              # dataset id
    "-npfp", "4",           # fingerprint processes
    "-np", "4",             # preprocessing processes
    "--verify_dataset_integrity"
]

plan_and_preprocess_entry()

Fingerprint extraction...
Dataset001_CAC
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [3.      0.38625 0.38625]. 
Current patch size: (np.int64(28), np.int64(256), np.int64(256)). 
Current median shape: [ 47. 500. 500.]
Attempting to find 3d_lowres config. 
Current spacing: [3.        0.3978375 0.3978375]. 
Current patch size: (np.int64(28), np.int64(256), np.int64(256)). 
Current median shape: [ 47.        485.4368932 485.4368932]
Attempting to

Preprocessing cases: 100%|██████████| 368/368 [00:43<00:00,  8.38it/s]


Configuration: 3d_fullres...
{'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [24, 256, 256], 'median_image_size_in_voxels': [47.0, 515.0, 515.0], 'spacing': [3.0, 0.375, 0.375], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.PlainConvUNet', 'arch_kwargs': {'n_stages': 7, 'features_per_stage': [32, 64, 128, 256, 320, 320, 320], 

Preprocessing cases: 100%|██████████| 368/368 [00:42<00:00,  8.61it/s]


Configuration: 3d_lowres...
INFO: Configuration 3d_lowres not found in plans file nnUNetPlans.json of dataset Dataset001_CAC. Skipping.


## Training nnUNet

In [5]:
import os

os.environ["TORCHDYNAMO_DISABLE"] = "1"    
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TRITON_CACHE_DIR"] = "/tmp/triton"

import torch
torch._dynamo.config.suppress_errors = True

In [6]:
import sys
from nnunetv2.run.run_training import run_training_entry

sys.argv = [
    "nnUNetv2_train",
    "1",              # dataset id
    "3d_fullres",     # configuration
    "0",              # fold (0–4)
]

run_training_entry()


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-08-05 16:26:26.480208: Using torch.compile...
2026-08-05 16:26:26.497787: do_dummy_2d_data_aug: True
2026-08-05 16:26:26.498396: Using splits from existing split file: /kaggle/working/nnUNet_preprocessed/Dataset001_CAC/splits_final.json
2026-08-05 16:26:26.498553: The split file contains 5 

## Test Set Results (FOLD 0 TRAINED MODEL)

In [7]:
import torch

torch.set_num_interop_threads = lambda *args, **kwargs: None

In [8]:
import sys
from nnunetv2.inference.predict_from_raw_data import predict_entry_point

sys.argv = [
    "nnUNetv2_predict", 
    "-i", "/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper/nnUNet_raw/Dataset001_CAC/imagesTs",
    "-o", "/kaggle/working/predictions",
    "-d", "Dataset001_CAC",
    "-c", "3d_fullres",
    "-f", "0",
]

predict_entry_point()


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 65 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 65 cases that I would like to predict

Predicting 0b4f988d88dc:
perform_everything_on_device: True


100%|██████████| 64/64 [00:06<00:00, 10.42it/s]


sending off prediction to background worker for resampling and export
done with 0b4f988d88dc

Predicting 0c6717ca6bb9:
perform_everything_on_device: True


100%|██████████| 48/48 [00:04<00:00, 10.45it/s]


sending off prediction to background worker for resampling and export
done with 0c6717ca6bb9

Predicting 1795b7f5218d:
perform_everything_on_device: True


...

100%|██████████| 27/27 [00:02<00:00, 10.49it/s]


sending off prediction to background worker for resampling and export
done with ff99d74f2dd7

Predicting ffa19985900c:
perform_everything_on_device: True


100%|██████████| 100/100 [00:09<00:00, 10.36it/s]


sending off prediction to background worker for resampling and export
done with ffa19985900c
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.


In [9]:
import os
import numpy as np
import SimpleITK as sitk

pred_dir = "/kaggle/working/predictions"
gt_dir = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper/nnUNet_raw/Dataset001_CAC/labelsTs"

# Only prediction images
files = sorted([
    f for f in os.listdir(pred_dir)
    if f.endswith(".nii.gz")
])

dice_scores = []

for f in files:

    pred_path = os.path.join(pred_dir, f)
    gt_path = os.path.join(gt_dir, f)

    if not os.path.exists(gt_path):
        print(f"Ground truth not found for {f}, skipping.")
        continue

    pred = sitk.GetArrayFromImage(sitk.ReadImage(pred_path))
    gt = sitk.GetArrayFromImage(sitk.ReadImage(gt_path))

    labels = sorted(set(np.unique(pred)).union(set(np.unique(gt))))
    labels = [l for l in labels if l != 0]  # Ignore background

    case_dices = []

    for label in labels:

        pred_mask = (pred == label)
        gt_mask = (gt == label)

        tp = np.logical_and(pred_mask, gt_mask).sum()
        fp = np.logical_and(pred_mask, ~gt_mask).sum()
        fn = np.logical_and(~pred_mask, gt_mask).sum()

        denom = 2 * tp + fp + fn

        if denom == 0:
            dice = 1.0
        else:
            dice = 2 * tp / denom

        case_dices.append(dice)

    mean_case_dice = np.mean(case_dices) if case_dices else 1.0

    dice_scores.append(mean_case_dice)

    print(f"{f:<35} Dice = {mean_case_dice:.4f}")

print("-" * 60)
print(f"Average Dice = {np.mean(dice_scores):.4f}")

0b4f988d88dc.nii.gz                 Dice = 0.9300
0c6717ca6bb9.nii.gz                 Dice = 0.8589
1795b7f5218d.nii.gz                 Dice = 0.0000
1acd2b512fd5.nii.gz                 Dice = 0.7846
1f8c171b782f.nii.gz                 Dice = 0.9026
22815eec22c1.nii.gz                 Dice = 0.8836
22fa117238e4.nii.gz                 Dice = 0.8587
2fba29dcc30a.nii.gz                 Dice = 0.9060
38222120d7dc.nii.gz                 Dice = 0.8743
3920827b9211.nii.gz                 Dice = 0.2778
3a66eef415c2.nii.gz                 Dice = 0.8919
3b8a40f8dc21.nii.gz                 Dice = 0.8624
3baa53642d99.nii.gz                 Dice = 0.9014
3efd39a9eb82.nii.gz                 Dice = 0.0726
40fe3962948a.nii.gz                 Dice = 0.9583
422e2fa73819.nii.gz                 Dice = 0.8369
4b0782a6242e.nii.gz                 Dice = 0.9001
52bd8b1109c6.nii.gz                 Dice = 0.8995
638bea82466a.nii.gz                 Dice = 0.8341
65bd2961b3b9.nii.gz                 Dice = 0.9610


In [10]:
import os
import shutil

pred_dir = "/kaggle/working/predictions"
gt_dir = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper/nnUNet_raw/Dataset001_CAC/labelsTs"
img_dir = "/kaggle/input/notebooks/kalabhalu/arc-nnunet-simple-scv-unzipper/nnUNet_raw/Dataset001_CAC/imagesTs"

save_dir = "/kaggle/working/Grouped"
os.makedirs(save_dir, exist_ok=True)

# predictions only
pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith(".nii.gz")])

for pred_file in pred_files:

    # -------- extract id --------
    # prediction: id.nii.gz OR id something
    case_id = pred_file.replace(".nii.gz", "")

    # ground truth format: id.nii.gz
    gt_file = f"{case_id}.nii.gz"

    # image format: id_0000.nii.gz
    img_file = f"{case_id}_0000.nii.gz"

    case_folder = os.path.join(save_dir, case_id)
    os.makedirs(case_folder, exist_ok=True)

    # -------- copy prediction --------
    shutil.copy(
        os.path.join(pred_dir, pred_file),
        os.path.join(case_folder, "prediction.nii.gz")
    )

    # -------- copy GT --------
    gt_path = os.path.join(gt_dir, gt_file)
    if os.path.exists(gt_path):
        shutil.copy(gt_path, os.path.join(case_folder, "label.nii.gz"))
    else:
        print("Missing GT:", gt_path)

    # -------- copy image --------
    img_path = os.path.join(img_dir, img_file)
    if os.path.exists(img_path):
        shutil.copy(img_path, os.path.join(case_folder, "image.nii.gz"))
    else:
        print("Missing image:", img_path)

print("DONE ✔ Grouped dataset created")

DONE ✔ Grouped dataset created
